# 03 — Tyre Degradation Modeling & Leave-One-Race-Out Validation

**Project:** F1 Tyre Degradation & Pit Strategy Optimizer  
**Primary Decision Question:** *"For a given Grand Prix, when is the optimal pit stop window and which tyre compound sequence minimizes total race time based on empirical tyre degradation?"*

---

## Stage 3 Objectives
1. **Baseline Model:** Fit linear regression per compound and circuit (`slope = degradation rate in seconds/lap`), documenting raw OLS, within-stint fixed-effects slopes, and empirical Bayes regularization for sparse traffic-confounded regimes (`Hungary` Soft).
2. **Expressive Models:** Train physics-informed non-linear and mixed-interaction models predicting degradation and lap time from `TyreLife`, `StintLapAge`, `Compound`, `TrackTemp`/`AirTemp`, and `EstimatedFuelKg`:
   - `RidgeMixedInteractionModel`: $L_2$-regularized linear mixed-interaction model with circuit lap-duration scaling.
   - `MonotonicBoostingDegradationModel`: `HistGradientBoostingRegressor` with positive monotonic wear constraints (`+1` on tyre age features) and robust $L_1$ loss.
   - `HybridPhysicsBoostingModel`: 50/50 ensemble combining smooth linear physical derivatives with non-linear monotonic tree boosting.
3. **Leak-Free Validation (5-Fold Leave-One-Race-Out CV):** Hold out one entire Grand Prix per fold as the test set and train strictly on the other four Grand Prix (no random lap splitting).
4. **Error Analysis & Honest Limitations:** Slice out-of-fold residuals across tyre age buckets, race traffic phases, compounds, and worst-predicted driver stints.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.load import DEFAULT_RACES_2024, RANDOM_SEED
from src.model import (
    fit_circuit_compound_baseline_table,
    fit_full_models,
    prepare_modeling_features,
    run_leave_one_race_out_cv,
    analyze_model_errors,
    run_stage3_pipeline,
)

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

COMPOUND_COLORS = {
    "SOFT": "#E8002D",
    "MEDIUM": "#E6B800",
    "HARD": "#555555",
}


## 1. Baseline Model: Linear Regression per Compound & Circuit

First, we execute the Stage 3 pipeline (`run_stage3_pipeline`) and inspect the 15-combination (`5 circuits × 3 compounds`) Baseline Linear Regression table.
- **`RawOLS_Slope_s_per_lap`**: Unconstrained OLS slope of `StintDegradationDelta ~ TyreLife`.
- **`WithinStintFE_Slope_s_per_lap`**: Demeaned within-stint fixed-effects slope (`FuelAdjustedLapTime ~ TyreLife + Driver:Stint`).
- **`Regularized_Slope_s_per_lap`**: Applies empirical Bayes pooling toward the global compound prior when sample size is small (`CleanLaps <= 35`) or confounded by Lap 2–6 DRS train release (`Hungary` Soft).


In [2]:
stage3_results = run_stage3_pipeline(save_outputs=True)

baseline_table = stage3_results["baseline_table"]
compound_offsets = stage3_results["compound_offsets"]
loro_metrics = stage3_results["loro_metrics"]
oof_df = stage3_results["oof_predictions"]
error_regimes = stage3_results["error_regimes"]
worst_stints = stage3_results["worst_stints"]

baseline_table


,Circuit,Compound,Stints,CleanLaps,RawOLS_Slope_s_per_lap,WithinStintFE_Slope_s_per_lap,Regularized_Slope_s_per_lap,Intercept_s,Correlation_r,InSample_R2,InSample_MAE_s,InSample_RMSE_s,UsedEmpiricalBayesPrior,MeanTrackTemp_C,FreshOffsetVsMediumSeconds
0,Abu Dhabi,MEDIUM,17,234,0.0730,0.0817,0.0730,-0.195,0.758,0.575,0.3010,0.3849,False,31.4,0.000
1,Abu Dhabi,HARD,24,620,0.0512,0.0557,0.0512,-0.073,0.782,0.612,0.3056,0.4033,False,30.4,0.193
2,Abu Dhabi,SOFT,2,24,0.1655,0.1678,0.1015,-0.126,0.878,0.771,0.5052,0.6037,True,29.9,-0.242
3,Hungary,SOFT,6,30,-0.0917,-0.0840,0.0851,-0.198,-0.561,0.314,0.3205,0.4857,True,46.5,-0.228
4,Hungary,HARD,30,774,0.0710,0.0780,0.0710,-0.027,0.681,0.463,0.5163,0.6835,False,44.7,0.182
5,Hungary,MEDIUM,23,379,0.0549,0.0647,0.0549,0.022,0.493,0.243,0.4855,0.6874,False,45.3,0.000
6,Japan,SOFT,8,48,0.1187,0.1202,0.1187,-0.254,0.704,0.496,0.3546,0.4357,False,35.8,-0.263
7,Japan,MEDIUM,17,215,0.0810,0.1001,0.0810,0.045,0.623,0.388,0.4037,0.5101,False,35.8,0.000
8,Japan,HARD,28,466,0.0855,0.0921,0.0855,0.112,0.636,0.405,0.5506,0.7000,False,34.9,0.210
9,Netherlands,MEDIUM,20,441,0.0459,0.0455,0.0459,0.035,0.620,0.385,0.3251,0.4545,False,31.0,0.000


In [3]:
fig_slopes = px.bar(
    baseline_table,
    x="Circuit",
    y="Regularized_Slope_s_per_lap",
    color="Compound",
    barmode="group",
    category_orders={
        "Circuit": list(DEFAULT_RACES_2024.keys()),
        "Compound": ["SOFT", "MEDIUM", "HARD"],
    },
    color_discrete_map=COMPOUND_COLORS,
    text="Regularized_Slope_s_per_lap",
    title="Soft Degrades at 0.057–0.119 s/lap vs. 0.041–0.086 s/lap for Hard Across 2024 Dry Circuits",
    labels={
        "Regularized_Slope_s_per_lap": "Degradation Rate (s / lap)",
        "Circuit": "Grand Prix Circuit",
        "Compound": "Tyre Compound",
    },
    template="plotly_white",
)
fig_slopes.update_traces(texttemplate="%{text:.3f}s", textposition="outside")
fig_slopes.update_layout(height=480, yaxis_range=[0, 0.14])
fig_slopes.show()


## 2. Leak-Free Validation: 5-Fold Leave-One-Race-Out Cross-Validation (LORO-CV)

Why **Leave-One-Race-Out CV** instead of random lap splitting?
- In Formula 1 telemetry, consecutive laps within the same driver stint share strong autocorrelation, track temperature, and car setup. A random 80/20 split across laps leaks mid-stint pace into the training set and artificially inflates accuracy.
- Holding out **one entire Grand Prix (`Circuit`) per fold** tests whether our physical scaling (`LapDurationScale = CircuitBaselinePace / 85.0`), fuel-load interaction, and monotonic tyre wear curves generalize to an **unseen race weekend**.


In [4]:
loro_metrics


,Fold,TestCircuit,TestLaps,Baseline_MAE_s,Baseline_RMSE_s,Ridge_MAE_s,Ridge_RMSE_s,MonotonicHGB_MAE_s,MonotonicHGB_RMSE_s,Hybrid_MAE_s,Hybrid_RMSE_s,MAE_Improvement_vs_Baseline_Pct,RMSE_Improvement_vs_Baseline_Pct
0,Fold 1,Abu Dhabi,878,0.3893,0.4764,0.3302,0.4363,0.3438,0.4538,0.3163,0.4228,18.75,11.27
1,Fold 2,Hungary,1183,0.5400,0.7353,0.5436,0.7313,0.5362,0.7316,0.5326,0.7235,1.38,1.60
2,Fold 3,Japan,729,0.5830,0.7675,0.5173,0.6715,0.5094,0.6812,0.5077,0.6691,12.93,12.82
3,Fold 4,Netherlands,1271,0.5066,0.6320,0.4270,0.5542,0.4204,0.5435,0.4190,0.5427,17.29,14.13
4,Fold 5,Spain,1135,0.4571,0.6110,0.4680,0.6162,0.4722,0.6387,0.4588,0.6159,-0.37,-0.80
5,Mean (5-Fold LORO),ALL 5 RACES,5196,0.4952,0.6444,0.4572,0.6019,0.4564,0.6098,0.4469,0.5948,9.76,7.70
6,Std Dev (5-Fold LORO),STD ACROSS FOLDS,0,0.0750,0.1150,0.0840,0.1135,0.0765,0.1112,0.0852,0.1171,8.95,6.89


In [5]:
folds_plot_df = loro_metrics[loro_metrics["Fold"].str.startswith("Fold")].copy()
melted_folds = folds_plot_df.melt(
    id_vars=["Fold", "TestCircuit"],
    value_vars=["Baseline_MAE_s", "Ridge_MAE_s", "MonotonicHGB_MAE_s", "Hybrid_MAE_s"],
    var_name="Model",
    value_name="MAE_Seconds",
)
model_label_map = {
    "Baseline_MAE_s": "1. Baseline OLS (Compound Linear)",
    "Ridge_MAE_s": "2. Regularized Ridge Mixed-Interaction",
    "MonotonicHGB_MAE_s": "3. Monotonic HistGradientBoosting",
    "Hybrid_MAE_s": "4. Hybrid Physics + Monotonic GBDT",
}
melted_folds["Model"] = melted_folds["Model"].map(model_label_map)

fig_loro = px.bar(
    melted_folds,
    x="TestCircuit",
    y="MAE_Seconds",
    color="Model",
    barmode="group",
    text="MAE_Seconds",
    title="Hybrid Physics-Guided Monotonic Boosting Reduces LORO-CV MAE by 9.8% Overall (Up to 18.8% on Unseen Circuits)",
    labels={
        "TestCircuit": "Held-Out Test Grand Prix (Leave-One-Race-Out Fold)",
        "MAE_Seconds": "Out-of-Fold Mean Absolute Error (seconds)",
    },
    template="plotly_white",
)
fig_loro.update_traces(texttemplate="%{text:.3f}s", textposition="outside")
fig_loro.update_layout(height=480, yaxis_range=[0, 0.68], legend_title_text="Model Architecture")
fig_loro.show()


## 3. Predicted Degradation Curves: Expressive Model vs. Linear Baseline

Below we compare the out-of-fold degradation curves predicted by the **Hybrid Physics-Guided Monotonic Boosting** model against the **Linear Baseline** and actual binned observations across all five held-out circuits.


In [6]:
binned_oof = (
    oof_df.groupby(["Circuit", "Compound", "TyreLife"], as_index=False)
    .agg(
        ActualDelta=("StintDegradationDelta", "median"),
        BaselinePred=("Pred_Baseline_Delta", "median"),
        HybridPred=("Pred_Hybrid_Delta", "median"),
        LapsCount=("LapNumber", "count"),
    )
)
binned_oof = binned_oof[binned_oof["LapsCount"] >= 2].copy()

circuits_order = list(DEFAULT_RACES_2024.keys())
fig_pred = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=circuits_order,
    shared_xaxes=False,
    shared_yaxes=True,
    vertical_spacing=0.14,
    horizontal_spacing=0.07,
)

for idx, circuit in enumerate(circuits_order):
    r = (idx // 3) + 1
    c = (idx % 3) + 1
    c_sub = binned_oof[binned_oof["Circuit"] == circuit]

    for comp in ["SOFT", "MEDIUM", "HARD"]:
        sub = c_sub[c_sub["Compound"] == comp].sort_values("TyreLife")
        if sub.empty:
            continue

        # Actual median observations
        fig_pred.add_trace(
            go.Scatter(
                x=sub["TyreLife"],
                y=sub["ActualDelta"],
                mode="markers",
                name=f"{comp} (Actual)",
                legendgroup=comp,
                showlegend=(idx == 0),
                marker=dict(color=COMPOUND_COLORS[comp], size=6, opacity=0.65),
            ),
            row=r,
            col=c,
        )
        # Out-of-fold Hybrid model curve
        fig_pred.add_trace(
            go.Scatter(
                x=sub["TyreLife"],
                y=sub["HybridPred"],
                mode="lines",
                name=f"{comp} (Hybrid LORO Pred)",
                legendgroup=comp,
                showlegend=False,
                line=dict(color=COMPOUND_COLORS[comp], width=2.8),
            ),
            row=r,
            col=c,
        )

    fig_pred.update_xaxes(title_text="Tyre Age (Laps)", row=r, col=c)
    if c == 1:
        fig_pred.update_yaxes(title_text="Degradation Delta (s)", row=r, col=c)

fig_pred.update_layout(
    title_text="Out-of-Fold Hybrid Model Predictions Track Empirical Wear Rates Across Short (76s) and Long (97s) Circuits",
    template="plotly_white",
    height=680,
)
fig_pred.show()


## 4. Error Analysis: Where Does the Model Struggle?

A rigorous sports analytics model must be transparent about its failure modes. We slice the 5,196 out-of-fold lap predictions across **Tyre Age Buckets**, **Race Phase / Traffic Buckets**, **Compounds**, and **Top 10 Worst-Predicted Driver Stints**.


In [7]:
error_regimes


,SliceType,Regime,Laps,Baseline_MAE_s,Hybrid_MAE_s,Baseline_RMSE_s,Hybrid_RMSE_s,Hybrid_SignedBias_s
0,TyreLifeBucket,1-5 Laps (Fresh / Warm-up),841,0.3257,0.2824,0.4341,0.4058,0.0031
1,TyreLifeBucket,6-15 Laps (Prime Stint Window),2220,0.4581,0.4296,0.6094,0.5842,0.0480
2,TyreLifeBucket,16-25 Laps (High Wear Window),1463,0.5557,0.5242,0.7110,0.6786,-0.0070
3,TyreLifeBucket,>25 Laps (Extended / Thermal Cliff),672,0.6910,0.5553,0.8428,0.7087,-0.0809
4,RacePhaseBucket,Laps 2-5 (Opening Pack / DRS Train),278,0.3133,0.2748,0.3885,0.3555,0.0118
5,RacePhaseBucket,Laps 6-40 (Mid-Race Strategic Phase),2923,0.4702,0.4280,0.6230,0.5820,0.0392
6,RacePhaseBucket,Laps >40 (Late-Race Low Fuel),1995,0.5548,0.5031,0.7162,0.6650,-0.0368
7,Compound,HARD,2917,0.5464,0.4740,0.7079,0.6333,-0.0137
8,Compound,MEDIUM,1650,0.4335,0.4110,0.5739,0.5615,0.0115
9,Compound,SOFT,629,0.4121,0.4300,0.5558,0.5881,0.1044


In [8]:
tyre_slice = error_regimes[error_regimes["SliceType"] == "TyreLifeBucket"].copy()

fig_err_age = go.Figure()
fig_err_age.add_trace(
    go.Bar(
        x=tyre_slice["Regime"],
        y=tyre_slice["Baseline_MAE_s"],
        name="Baseline Linear MAE",
        marker_color="#9E9E9E",
        text=tyre_slice["Baseline_MAE_s"],
        texttemplate="%{text:.3f}s",
        textposition="outside",
    )
)
fig_err_age.add_trace(
    go.Bar(
        x=tyre_slice["Regime"],
        y=tyre_slice["Hybrid_MAE_s"],
        name="Hybrid Physics+GBDT MAE",
        marker_color="#E8002D",
        text=tyre_slice["Hybrid_MAE_s"],
        texttemplate="%{text:.3f}s",
        textposition="outside",
    )
)
fig_err_age.update_layout(
    title="Prediction Error Doubles Beyond Lap 25 Due to Non-Linear Thermal Cliff & Traffic, Though Hybrid Cuts Cliff Error by 19.6%",
    xaxis_title="Tyre Age Regime",
    yaxis_title="Out-of-Fold MAE (seconds)",
    yaxis_range=[0, 0.80],
    barmode="group",
    template="plotly_white",
    height=460,
)
fig_err_age.show()


### 4.1 Diagnosis of Top 10 Highest-Error Driver Stints


In [9]:
worst_stints


,Circuit,Driver,Team,Stint,Compound,CleanLaps,StartLap,EndLap,MaxTyreLife,Baseline_MAE_s,Hybrid_MAE_s,MeanSignedError_s
0,Spain,BOT,Kick Sauber,2,SOFT,15,13,27,19,1.355,1.464,1.415
1,Hungary,BOT,Kick Sauber,3,HARD,23,47,69,24,1.337,1.305,1.101
2,Japan,HUL,Haas F1 Team,3,HARD,25,7,32,27,1.494,1.214,1.112
3,Hungary,BOT,Kick Sauber,1,MEDIUM,14,2,15,16,0.988,1.211,-1.191
4,Japan,TSU,RB,3,HARD,13,9,21,14,1.408,1.201,1.086
5,Spain,RIC,RB,3,HARD,21,44,65,23,1.042,1.178,-0.990
6,Hungary,STR,Aston Martin,1,SOFT,10,2,12,15,1.060,1.157,-1.062
7,Hungary,VER,Red Bull Racing,3,MEDIUM,19,51,70,21,1.167,1.154,1.094
8,Hungary,TSU,RB,1,MEDIUM,27,2,28,28,0.960,1.085,-0.994
9,Japan,BOT,Kick Sauber,3,HARD,12,8,21,15,1.240,1.024,0.884


### Root-Cause Analysis of High-Error Regimes & Honest Model Limitations

1. **Late-Stint Thermal Cliff & Driver Management Variance (`TyreLife > 25 laps`, MAE = `0.555 s` vs `0.282 s` at `1-5 laps`):**
   - Beyond 25 laps on a single set, tyres approach the carcass/thermal wear cliff where micro-blistering or graining causes sudden non-linear pace drop-offs. While the Hybrid model reduces MAE in this regime by **19.6%** compared to the linear baseline (`0.555 s` vs `0.691 s`), individual car setup and driver lift-and-coast management create substantial variance.
2. **Midfield DRS Trains & Dirty Air (`Bottas Spain Stint 2 SOFT`, `Hulkenberg Japan Stint 3 HARD`):**
   - Our features do not include real-time gap-to-car-ahead (`IntervalToPositionAhead`) because during pre-race strategy simulation (Stage 4) future traffic gaps are unknown. When a driver pits into a 4-car midfield DRS train (e.g., Bottas on Softs in Spain losing `+1.415 s/lap` relative to clean-air expectation), the model under-predicts their lap time.
3. **Opening-Pack Congestion Release (`Stroll Hungary Stint 1 SOFT`, `Tsunoda Hungary Stint 1 MEDIUM`):**
   - In Lap 2–6 at tight circuits like Hungaroring, cars outside the top 5 are bottlenecked by the pack (`~0.8–1.0 s/lap` slower than free air). Once early stoppers pit on Lap 6–8, remaining drivers experience a sudden `-1.0 s` lap time drop despite older tyres (`MeanSignedError = -1.06 s`).
4. **On-Track Incidents Not Flagged by SC/VSC (`Verstappen Hungary Stint 3 MEDIUM`):**
   - Verstappen's Stint 3 at Hungary includes his battle and Turn 1 lock-up/collision with Hamilton on Lap 63 (`MeanSignedError = +1.094 s`), which did not trigger a Safety Car and stayed within the 107% cutoff.
5. **Unobserved Pirelli C-Compound Step Nominations (`C1`–`C5`):**
   - FastF1 exposes relative race labels (`SOFT`, `MEDIUM`, `HARD`) rather than Pirelli's absolute compound code (`C1`–`C5`). While `LapDurationScale` and `TrackTemp` capture part of the circuit severity, a C3 Soft at Suzuka is physically harder than a C5 Soft at Montreal/Monza.
